# Requesting ATNF observations: Murriyang and ASKAP

This example shows how to build, validate and submit observation requests for
two telescopes run by CSIRO's Australia Telescope National Facility (ATNF):
Murriyang (the Parkes radio telescope) and ASKAP (the Australian Square
Kilometre Array Pathfinder).

The observatory cannot be called from outside. Requests are queued on a relay
that both sides can reach, and the observatory polls the relay and turns each
request into a Scheduling Block (SB), the set of parameters that defines an
observation. Submitting therefore returns a request id at once, and you poll
the request for progress.

## Configuration

Submitting needs two settings per telescope, read by `aeonlib.conf.Settings`:

1. The relay address, `murriyang_api_root` or `askap_api_root`
   (`AEON_MURRIYANG_API_ROOT`, `AEON_ASKAP_API_ROOT`).
2. Your ATNF observing token, `murriyang_token` or `askap_token`
   (`AEON_MURRIYANG_TOKEN`, `AEON_ASKAP_TOKEN`). ATNF issues it for your
   project, and the observatory checks each request against the allocation it
   describes.

Building and validating a request needs neither, so every cell up to
[Submit the request](#Submit-the-request) runs without them.

In [1]:
import json

from pydantic import TypeAdapter, ValidationError

from aeonlib.atnf.askap.models import AskapStandard
from aeonlib.atnf.murriyang.models import MURRIYANG_TRACKS, CryopafTrack, UwlTrack
from aeonlib.conf import settings
from aeonlib.models import SiderealTarget

## Create a Murriyang request

Murriyang supports tracking a single sidereal (ICRS) target, with the UWL
(ultra-wideband low) receiver or the CryoPAF (cryogenically cooled phased array
feed) receiver. Each receiver has its own class, which offers only the settings
that apply to it.

Only `target` and `duration` (seconds) are required. Every other field overrides
the observatory's default for that receiver, so leave it unset unless you need
it.

In [2]:
target = SiderealTarget(name="S9", type="ICRS", ra="17h52m05.5s", dec="-34d25m41.8s")

track = UwlTrack(target=target, duration=300)
track

UwlTrack(target=SiderealTarget(name='S9', type='ICRS', hour_angle=None, ra=<Angle 17.86819444 hourangle>, dec=<Angle -34.42827778 deg>, altitude=None, azimuth=None, proper_motion_ra=0, proper_motion_dec=0, epoch=2000, parallax=0), duration=300.0, receiver='uwl', sky_frequency=None, cal_signal=None)

`to_sb_params` shows the SB parameters the request sets. Fields left unset are
not included, so the observatory's defaults apply to them. The target becomes a
field name and a `[ra, dec, frame]` direction in decimal degrees.

In [3]:
track.to_sb_params()

{'duration': 300.0,
 'field_name': 'S9',
 'field_direction': [268.0229166666666, -34.42827777777777, 'J2000']}

A CryoPAF track offers more settings, such as the footprint, the pattern of
beams formed on the sky.

In [4]:
paf_track = CryopafTrack(
    target=target,
    duration=600,
    sky_frequency=1420.4,
    footprint={"name": "hextile72", "pitch": 0.162},
)
paf_track.to_sb_params()

{'duration': 600.0,
 'sky_frequency': 1420.4,
 'footprint': {'name': 'hextile72', 'pitch': 0.162},
 'field_name': 'S9',
 'field_direction': [268.0229166666666, -34.42827777777777, 'J2000']}

Each class lists its fields, with descriptions, units and limits, in its JSON
schema.

In [5]:
for name, field in CryopafTrack.model_json_schema()["properties"].items():
    print(f"{name}: {field.get('description', '')} {field.get('unit', '')}")

target:  
duration: Integration time s
receiver:  
sky_frequency: Sky Frequency at the center of the band MHz
filter_band: Frequency filter band select 
pol_axis: Polarization axis mode and value 
cal_signal: control calibration signal 
footprint:  
stream: which data stream to use 
stream_band: which data stream frequency band to use 


## Validate the request

Limits are checked when the request is built, so a bad value fails at once,
before anything is sent.

In [6]:
try:
    UwlTrack(target=target, duration=5)
except ValidationError as error:
    print(error)

1 validation error for UwlTrack
duration
  Input should be greater than or equal to 10 [type=greater_than_equal, input_value=5, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal


In [7]:
try:
    CryopafTrack(target=target, duration=600, footprint={"name": "hextile72"})
except ValidationError as error:
    print(error)

1 validation error for CryopafTrack
footprint.pitch
  Field required [type=missing, input_value={'name': 'hextile72'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


Only ICRS targets without proper motion are supported. `validate_observation`
on a facility, or `to_sb_params` directly, rejects other targets without any
network call.

In [8]:
moving = SiderealTarget(
    name="PSR B1133+16",
    type="ICRS",
    ra=174.0135,
    dec=15.8513,
    proper_motion_ra=-73.95,
    proper_motion_dec=368.05,
)
try:
    UwlTrack(target=moving, duration=300).to_sb_params()
except ValueError as error:
    print(error)

ATNF directions carry no proper motion, apply it to the coordinates first


## What is sent

The request is sent as JSON. The `receiver` value picks the class, so the
receiving side can parse any Murriyang track with `MURRIYANG_TRACKS`.

In [9]:
body = paf_track.model_dump(mode="json", exclude_none=True)
print(json.dumps(body, indent=2))

{
  "target": {
    "name": "S9",
    "type": "ICRS",
    "ra": 268.0229166666666,
    "dec": -34.42827777777777,
    "proper_motion_ra": 0.0,
    "proper_motion_dec": 0.0,
    "epoch": 2000,
    "parallax": 0.0
  },
  "duration": 600.0,
  "receiver": "cryopaf",
  "sky_frequency": 1420.4,
  "footprint": {
    "name": "hextile72",
    "pitch": 0.162
  }
}


In [10]:
TypeAdapter(MURRIYANG_TRACKS).validate_python(body)

CryopafTrack(target=SiderealTarget(name='S9', type='ICRS', hour_angle=None, ra=<Angle 268.02291667 deg>, dec=<Angle -34.42827778 deg>, altitude=None, azimuth=None, proper_motion_ra=0.0, proper_motion_dec=0.0, epoch=2000, parallax=0.0), duration=600.0, receiver='cryopaf', sky_frequency=1420.4, filter_band=None, pol_axis=None, cal_signal=None, footprint=FootprintParams(name='hextile72', pitch=0.162), stream=None, stream_band=None)

## Create an ASKAP request

`AskapStandard` observes one sidereal (ICRS) target, for example a target of
opportunity. As for Murriyang, only `target` and `duration` are required.
`align_to_beam` shifts the field direction so the target sits in the centre of
the given beam.

In [11]:
too = AskapStandard(
    target=SiderealTarget(name="GRB261010A", type="ICRS", ra=248.749083, dec=26.580194),
    duration=21600,
    align_to_beam=0,
)
too.to_sb_params()

{'duration': 21600.0,
 'align_to_beam': 0,
 'field_name': 'GRB261010A',
 'field_direction': [248.749083, 26.580194, 'J2000']}

## Submit the request

Submitting needs the configuration above, so this cell does nothing until it
is set.

`submit_observation` queues the request and returns its id.
`get_observation_status` reports its state: `PENDING` until the observatory
picks it up, then a final state such as `OBSERVED` or `FAILED`.
`cancel_observation` works until the observatory picks the request up.

With `dry_run=True` the observatory builds the parset, the text file of SB
parameters, without submitting anything; read it from `status.parset`. Pass a
timezone-aware `expires=` to stop a request being acted on if it waits too
long, which matters for a target of opportunity.

In [12]:
from datetime import datetime, timedelta, timezone
from time import sleep

from aeonlib.atnf.murriyang.facility import MurriyangFacility

if settings.murriyang_api_root and settings.murriyang_token:
    facility = MurriyangFacility()
    request_id = facility.submit_observation(
        paf_track,
        dry_run=True,
        expires=datetime.now(timezone.utc) + timedelta(hours=1),
    )
    print(f"Queued as {request_id}")

    status = facility.get_observation_status(request_id)
    while not status.terminal:
        sleep(10)
        status = facility.get_observation_status(request_id)
    print(status.state, status.message or "")
    print(status.parset)
else:
    print("Set AEON_MURRIYANG_API_ROOT and AEON_MURRIYANG_TOKEN to submit.")

Set AEON_MURRIYANG_API_ROOT and AEON_MURRIYANG_TOKEN to submit.
